# PolyChain: Polymer Property Prediction

**Architecture:** GIN-S backbone + HAMF + PECGN + CST

**Run time:** ~30-60 min for full 5-fold CV on T4 GPU.

**Instructions:** Run cells in order. Cell 1 is always required.

In [ ]:
#@title 1. Mount Drive + Clone + Install + Create Dirs
import os, sys
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"
for sub in ("checkpoints", "predictions", "reports", "results"):
    os.makedirs(f"{DRIVE_PATH}/{sub}", exist_ok=True)

# Clone repo
if not os.path.exists("/content/Poly/polymer_competition/features"):
    if os.path.exists("/content/Poly"):
        !rm -rf /content/Poly
    !git clone https://github.com/NotShubham1112/Poly.git /content/Poly

# Detect project root
if os.path.exists("/content/Poly/polymer_competition/features"):
    PROJECT_ROOT = "/content/Poly/polymer_competition"
elif os.path.exists("/content/Poly/features"):
    PROJECT_ROOT = "/content/Poly"
else:
    candidates = [os.getcwd(), "/content/Poly/polymer_competition", "/content/Poly"]
    for c in candidates:
        if os.path.exists(os.path.join(c, "features")):
            PROJECT_ROOT = c
            break
    else:
        raise FileNotFoundError("Could not find project root with features/ directory")

os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Create all required directories
ROOT = Path(PROJECT_ROOT)
for d in ["outputs", "outputs/checkpoints", "outputs/logs",
          "predictions", "reports", "reports/plots", "results",
          "data/processed"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

# Install dependencies
print("Installing PyTorch + PyG...")
!pip install -q torch torchvision torch-geometric
print("Installing RDKit + tree models...")
!pip install -q rdkit xgboost lightgbm catboost
print("Installing scikit-learn + pandas + utils...")
!pip install -q scikit-learn pandas numpy scipy pyyaml tqdm
print("Installing matplotlib + seaborn...")
!pip install -q matplotlib seaborn joblib

# Verify
import torch
import rdkit
import torch_geometric
print(f"PyTorch: {torch.__version__}")
print(f"PyG: {torch_geometric.__version__}")
print(f"RDKit: {rdkit.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")
else:
    print("GPU: not available (using CPU)")
print(f"Project root: {PROJECT_ROOT}")
print("Setup complete!")

In [ ]:
#@title 2. Environment Verification
import sys
from pathlib import Path
import torch

def verify_environment():
    checks = [
        ("torch", "torch"),
        ("torch_geometric", "torch_geometric"),
        ("rdkit", "rdkit"),
        ("numpy", "numpy"),
        ("pandas", "pandas"),
        ("scipy", "scipy"),
        ("sklearn", "sklearn"),
        ("xgboost", "xgboost"),
        ("lightgbm", "lightgbm"),
        ("catboost", "catboost"),
        ("matplotlib", "matplotlib"),
        ("yaml", "yaml"),
    ]
    failed = []
    for name, import_name in checks:
        try:
            mod = __import__(import_name)
            ver = getattr(mod, "__version__", "unknown")
            print(f"  {name:20s} {ver}")
        except ImportError:
            print(f"  {name:20s} MISSING")
            failed.append(name)
    if failed:
        print(f"ERROR: Missing: {', '.join(failed)}. Run cell 1 first.")
        return False
    print(f"\n  CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("  WARNING: CPU mode (training will be slow)")
    print("\n  Environment check: PASSED")
    return True

ENV_OK = verify_environment()

In [ ]:
#@title 3. Experiment Configuration
#@markdown Choose experiment mode:
EXPERIMENT_MODE = "smoke" #@param ["smoke", "dev", "full"]
FOLD = 0  # fold index (0-based)
PERSON = "colab"  # checkpoint prefix
if EXPERIMENT_MODE == "smoke":
    MAX_SAMPLES = 100
    EPOCHS = 3
    MODELS = "polychain"
    ABLATION_EPOCHS = 3
elif EXPERIMENT_MODE == "dev":
    MAX_SAMPLES = 500
    EPOCHS = 20
    MODELS = "ridge,polychain"
    ABLATION_EPOCHS = 20
else:
    MAX_SAMPLES = None
    EPOCHS = 50
    MODELS = "ridge,rf,xgb,gcn,gat,polychain"
    ABLATION_EPOCHS = 50
print(f"Mode: {EXPERIMENT_MODE}")
print(f"  Samples: {MAX_SAMPLES or 'all'}")
print(f"  Epochs : {EPOCHS}")
print(f"  Person : {PERSON}")
print(f"  Fold   : {FOLD}")
print(f"  Models : {MODELS}")
print(f"  Ablation: {ABLATION_EPOCHS} epochs")

In [ ]:
#@title 4. Load or Download Dataset
import os, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
(ROOT / "data").mkdir(exist_ok=True)
train_path = ROOT / "data" / "train.csv"

if not train_path.exists():
    print("train.csv not found. Generating polymer Tg dataset...")
    !python data/download_polymer_data.py --dataset polymer_tg --data_dir data/

if not train_path.exists():
    raise FileNotFoundError(f"train.csv not found at {train_path}")

train_df = pd.read_csv(train_path)
print(f"Dataset: {len(train_df)} samples")
print(f"Columns: {list(train_df.columns)}")

for col in ["SMILES", "property"]:
    if col not in train_df.columns:
        raise ValueError(f"Column '{col}' missing. Found: {list(train_df.columns)}")

if len(train_df) == 0:
    raise ValueError("Empty dataset")

null_smiles = train_df["SMILES"].isna().sum()
if null_smiles:
    print(f"  Dropping {null_smiles} rows with null SMILES")
    train_df = train_df.dropna(subset=["SMILES"]).reset_index(drop=True)

if "property" in train_df.columns:
    y = train_df["property"]
    print(f"\nTarget: mean={y.mean():.3f}, std={y.std():.3f}")
    print(f"        min={y.min():.3f}, max={y.max():.3f}")
train_df.head()

In [ ]:
#@title 5. Visualize Target Distribution
import os, sys, pandas as pd
from pathlib import Path
from reports.visualizations import ReportGenerator

ROOT = Path.cwd()
train_path = ROOT / "data" / "train.csv"
if not train_path.exists():
    print("train.csv not found. Run cell 4 first.")
else:
    train_df = pd.read_csv(train_path)
    gen = ReportGenerator(str(ROOT / "reports" / "plots"))
    if "property" in train_df.columns:
        try:
            gen.plot_target_distribution(
                train_df["property"].values,
                target_name="property",
                save_name="target_distribution",
            )
            print("Plot: reports/plots/target_distribution.png")
        except Exception as e:
            print(f"Warning: plot failed: {e}")

In [ ]:
#@title 6. Generate CV Splits + Features
from pathlib import Path
ROOT = Path.cwd()
(ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)

feat_path = ROOT / "data" / "processed" / "train_features.parquet"
if feat_path.exists():
    print("Features exist. Delete data/processed/ to regenerate.")
else:
    print("Generating CV splits...")
    !python -m data.generate_splits --config config.yaml
    print("\nBuilding features...")
    !python -m features.build_features --config config.yaml

if not feat_path.exists():
    raise FileNotFoundError(f"Feature building failed: {feat_path}")
splits_path = ROOT / "data" / "splits.pkl"
if not splits_path.exists():
    raise FileNotFoundError(f"Splits missing: {splits_path}")
print(f"OK: {feat_path.name}, {splits_path.name}")

In [ ]:
#@title 7. Verify All Project Imports
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

modules = [
    "features.graphs",
    "features.graph_utils",
    "models.polychain",
    "models.polychain.cst",
    "models.gnn",
    "models.mlp",
    "training.train_utils",
    "reports.visualizations",
    "inference.predictor",
]

ok = True
for m in modules:
    try:
        __import__(m)
        print(f"  {m:35s} OK")
    except Exception as e:
        print(f"  {m:35s} FAIL - {e}")
        ok = False

if ok:
    print(f"\nAll imports OK!")
else:
    print(f"\nSome imports failed.")

In [ ]:
#@title 8. Smoke Test (train a model)
import sys, subprocess
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

feat_path = ROOT / "data" / "processed" / "train_features.parquet"
if not feat_path.exists():
    print("Features not found. Run cell 6 first.")
else:
    cmd = [sys.executable, "-m", "training.train",
           "--model_type", "polychain", "--fold", str(FOLD),
           "--person", PERSON, "--config", "config.yaml"]
    if MAX_SAMPLES:
        cmd += ["--max_samples", str(MAX_SAMPLES)]
    cmd += ["--epochs", str(EPOCHS)]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(ROOT))
    if result.returncode != 0:
        print(f"Training failed (code {result.returncode})")
    else:
        print("Training completed.")

In [ ]:
#@title 9. Verify Checkpoint + Inference
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ckpt_path = ROOT / "outputs" / "checkpoints" / f"{PERSON}_polychain_fold{FOLD}_best.pt"
if not ckpt_path.exists():
    print("No checkpoint. Run cell 8 first.")
else:
    try:
        from inference.predictor import PolymerPredictor
        predictor = PolymerPredictor(str(ckpt_path))
        test_smiles = ["*CCO*", "*CC(*)c1ccccc1*", "*CCl*"]
        preds = predictor.predict(test_smiles)
        print(f"{'SMILES':<30} {'Predicted':>10}")
        print("-" * 42)
        for smi, pred in zip(test_smiles, preds):
            print(f"{smi:<30} {pred:.4f}")
    except Exception as e:
        print(f"Inference error: {e}")

In [ ]:
#@title 10. Generate Training Plots
import json
from pathlib import Path
from reports.visualizations import ReportGenerator

ROOT = Path.cwd()
gen = ReportGenerator(str(ROOT / "reports" / "plots"))
metrics_file = ROOT / "outputs" / "logs" / "metrics.json"

if metrics_file.exists():
    try:
        with open(metrics_file) as f:
            history = json.load(f)
        if len(history) > 1:
            train_losses = [h.get("train_loss", 0) for h in history]
            val_losses = [h.get("val_loss", 0) for h in history]
            gen.plot_training_curves(train_losses, val_losses,
                                      title="PolyChain Training",
                                      save_name="training_curve_polychain")
            print("Plot: reports/plots/training_curve_polychain.png")
        else:
            print("Not enough epochs.")
    except Exception as e:
        print(f"Warning: {e}")
else:
    print("No metrics. Run cell 8 first.")

In [ ]:
#@title 11. Copy Checkpoint to Drive
import os, shutil
from pathlib import Path
ROOT = Path.cwd()
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"

ckpt_dir = ROOT / "outputs" / "checkpoints"
pattern = f"{PERSON}_polychain_fold{FOLD}_*.pt"
matches = list(ckpt_dir.glob(pattern))
if matches:
    dest = Path(DRIVE_PATH) / "checkpoints"
    dest.mkdir(parents=True, exist_ok=True)
    for f in matches:
        shutil.copy2(f, dest / f.name)
        print(f"Copied {f.name}")
else:
    print("No checkpoints to copy.")

d = f"{DRIVE_PATH}/checkpoints/"
print("Drive:", os.listdir(d) if os.path.exists(d) else [])

In [ ]:
#@title 12. Full 5-Fold CV
import sys, subprocess
from pathlib import Path

ROOT = Path.cwd()
print(f"Models: {MODELS}   Mode: {EXPERIMENT_MODE}")

cmd = [sys.executable, "-m", "training.run_all_folds",
       "--models", MODELS.replace(" ", ""),
       "--person", PERSON, "--config", "config.yaml"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(ROOT))
if result.returncode != 0:
    print(f"CV failed (code {result.returncode})")
else:
    print("5-fold CV complete.")

In [ ]:
#@title 13. View Results Summary
import pandas as pd
from pathlib import Path
ROOT = Path.cwd()
summary_path = ROOT / "results" / "summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
    print("=" * 60)
    print("  MODEL COMPARISON (5-Fold CV)")
    print("=" * 60)
    print(summary.to_string(index=False))
else:
    print("No summary. Run cell 12 first.")

In [ ]:
#@title 14. Model Comparison Plot
import sys, pandas as pd
from pathlib import Path
from reports.visualizations import ReportGenerator

ROOT = Path.cwd()
summary_path = ROOT / "results" / "summary.csv"
if summary_path.exists():
    try:
        gen = ReportGenerator(str(ROOT / "reports" / "plots"))
        summary = pd.read_csv(summary_path)
        if "rmse_mean" in summary.columns:
            model_rmse = dict(zip(summary["model"], summary["rmse_mean"]))
            gen.plot_model_comparison(model_rmse, metric_name="rmse",
                                       save_name="model_comparison")
            print("Plot: reports/plots/model_comparison.png")
    except Exception as e:
        print(f"Warning: {e}")
else:
    print("No summary. Run cell 12 first.")

In [ ]:
#@title 15. PolyChain Ablation Study
import sys, subprocess
from pathlib import Path
ROOT = Path.cwd()
print(f"Ablation: {ABLATION_EPOCHS} epochs")
cmd = [sys.executable, "-m", "training.run_ablation",
       "--fold", str(FOLD), "--epochs", str(ABLATION_EPOCHS),
       "--config", "config.yaml"]
result = subprocess.run(cmd, cwd=str(ROOT))
if result.returncode != 0:
    print(f"Ablation failed (code {result.returncode})")
else:
    print("Ablation complete.")

In [ ]:
#@title 16. View Ablation Results
import sys, pandas as pd
from pathlib import Path
from reports.visualizations import ReportGenerator

ROOT = Path.cwd()
ablation_path = ROOT / "results" / "ablation_results.csv"
if ablation_path.exists():
    try:
        gen = ReportGenerator(str(ROOT / "reports" / "plots"))
        ablation = pd.read_csv(ablation_path)
        print("=" * 60)
        print("  ABLATION STUDY")
        print("=" * 60)
        print(ablation.to_string(index=False))
        if "variant" in ablation.columns and "rmse" in ablation.columns:
            variant_rmse = dict(zip(ablation["variant"], ablation["rmse"]))
            gen.plot_ablation(variant_rmse, save_name="ablation")
            print("Plot: reports/plots/ablation.png")
    except Exception as e:
        print(f"Warning: {e}")
else:
    print("No ablation. Run cell 15 first.")

In [ ]:
#@title 17. Display Generated Plots
import os
from pathlib import Path
ROOT = Path.cwd()
plot_dir = ROOT / "reports" / "plots"
if plot_dir.exists():
    plots = sorted([f for f in os.listdir(str(plot_dir)) if f.endswith(".png")])
    if plots:
        print(f"{len(plots)} plots:")
        for p in plots:
            print(f"  - {p}")
    else:
        print("No plots yet.")
else:
    print("Plot directory missing.")

In [ ]:
#@title 18. Save Everything to Drive
import os
from pathlib import Path
ROOT = Path.cwd()
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"

for sub in ("checkpoints", "predictions", "reports", "results"):
    os.makedirs(f"{DRIVE_PATH}/{sub}", exist_ok=True)

if list((ROOT / "outputs" / "checkpoints").glob("*.pt")):
    !cp -rv "$ROOT/outputs/checkpoints/"*.pt "$DRIVE_PATH/checkpoints/"
if list((ROOT / "predictions").glob("*.pkl")):
    !cp -rv "$ROOT/predictions/"*.pkl "$DRIVE_PATH/predictions/"
if list((ROOT / "reports" / "plots").glob("*.png")):
    !cp -rv "$ROOT/reports/plots/"*.png "$DRIVE_PATH/reports/"
if list((ROOT / "results").glob("*.csv")):
    !cp -rv "$ROOT/results/"*.csv "$DRIVE_PATH/results/"

print(f"Synced to {DRIVE_PATH}")
for sub in ("checkpoints", "predictions", "reports", "results"):
    d = f"{DRIVE_PATH}/{sub}"
    files = os.listdir(d) if os.path.exists(d) else []
    print(f"  {sub}/: {len(files)} files")

In [ ]:
#@title 19. Resume After Disconnection
import os, subprocess, sys, shutil
from pathlib import Path
ROOT = Path.cwd()
DRIVE_PATH = "/content/drive/MyDrive/PolyChain"

drive_ckpts = Path(DRIVE_PATH) / "checkpoints"
if drive_ckpts.exists():
    for f in drive_ckpts.glob("*.pt"):
        dest = ROOT / "outputs" / "checkpoints" / f.name
        shutil.copy2(f, dest)
        print(f"Restored {f.name}")
final = ROOT / "outputs" / "checkpoints" / f"{PERSON}_polychain_fold{FOLD}_final.pt"
if final.exists():
    print("Resuming PolyChain...")
    cmd = [sys.executable, "-m", "training.train",
           "--model_type", "polychain", "--fold", str(FOLD),
           "--person", PERSON, "--config", "config.yaml", "--resume"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT))
else:
    print("No checkpoint to resume.")

In [ ]:
#@title 20. Inference Demo
import sys, glob
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

best_ckpts = sorted(glob.glob(str(ROOT / "outputs" / "checkpoints" / "*polychain*best.pt")))
if not best_ckpts:
    print("No checkpoint. Run training first.")
else:
    try:
        from inference.predictor import PolymerPredictor
        predictor = PolymerPredictor(best_ckpts[0])
        test = {
            "Polyethylene": "*CC*",
            "Polystyrene": "*CC(*)c1ccccc1*",
            "PVC": "*CCl*",
            "PMMA": "*CC(=O)OC*",
            "Nylon 6,6": "*CC(=O)NCCCCCC(=O)N*",
        }
        print(f"{'Polymer':<20} {'SMILES':<30} {'Predicted':>10}")
        print("-" * 62)
        for name, smi in test.items():
            pred = predictor.predict([smi])
            print(f"{name:<20} {smi:<30} {pred[0]:>10.2f}")
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
#@title 21. Auto Test Suite
import sys
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def test_notebook():
    passed, failed = 0, 0
    results = []

    def check(name, ok, detail=""):
        nonlocal passed, failed
        if ok:
            results.append((name, "PASS", ""))
            passed += 1
        else:
            results.append((name, "FAIL", detail))
            failed += 1

    # 1. Project imports
    try:
        from features.graphs import smiles_to_graph, BOND_FEAT_DIM
        from features.graph_utils import build_multiscale, collate_multiscale
        from models.polychain import PolyChain
        check("Project imports", True)
    except Exception as e:
        check("Project imports", False, str(e))

    # 2. Graph construction
    try:
        g = smiles_to_graph("*CCO*")
        check("smiles_to_graph", g is not None and g.x.size(1) > 0, str(g.x.shape))
        check("Graph edge_attr dim", g.edge_attr.size(1) == BOND_FEAT_DIM, str(g.edge_attr.shape))
    except Exception as e:
        check("Graph construction", False, str(e))

    # 3. Multiscale graph
    try:
        ms = build_multiscale("*CCO*")
        check("build_multiscale", ms is not None)
        check("Monomer+Dimer+Trimer", hasattr(ms, "monomer") and hasattr(ms, "dimer"))
    except Exception as e:
        check("Multiscale", False, str(e))

    # 4. Collate
    try:
        ms_list = [m for s in ["*CCO*", "*CCl*"] if (m := build_multiscale(s)) is not None]
        batch = collate_multiscale(ms_list)
        check("collate_multiscale", "monomer" in batch and "y" in batch)
    except Exception as e:
        check("collate", False, str(e))

    # 5. PolyChain forward
    try:
        from models.polychain.cst import compute_cst_batch
        model = PolyChain(in_atom_dim=BOND_FEAT_DIM, in_edge_dim=BOND_FEAT_DIM,
                          hidden_dim=32, n_backbone_layers=2, n_hamf_layers=1, cst_dim=32)
        batch["cst"] = torch.tensor(compute_cst_batch([s.smiles for s in ms_list]), dtype=torch.float)
        out = model(batch)
        check("PolyChain forward", out.shape == (len(ms_list),), str(out.shape))
    except Exception as e:
        check("PolyChain forward", False, str(e))

    # 6. Checkpoint save/load
    try:
        from training.train_utils import save_checkpoint, load_checkpoint
        tmp = ROOT / "outputs" / "test_ckpt.pt"
        save_checkpoint({"test": True, "model_state": model.state_dict()}, str(tmp))
        check("Save checkpoint", tmp.exists())
        data = load_checkpoint(str(tmp))
        check("Load checkpoint", data.get("test") is True)
        tmp.unlink(missing_ok=True)
    except Exception as e:
        check("Checkpoint save/load", False, str(e))

    # 7. Predictor
    try:
        from inference.predictor import PolymerPredictor
        from training.train_utils import save_checkpoint
        tmp = ROOT / "outputs" / "test_pred_ckpt.pt"
        cfg = {"in_atom_dim": BOND_FEAT_DIM, "in_edge_dim": BOND_FEAT_DIM,
               "hidden_dim": 32, "n_backbone_layers": 2, "n_hamf_layers": 1}
        save_checkpoint({"config": cfg, "model_state": model.state_dict(),
                         "cst_mean": np.zeros(32).tolist(), "cst_std": np.ones(32).tolist()}, str(tmp))
        pred = PolymerPredictor(str(tmp))
        result = pred.predict(["*CCO*"])
        check("PolymerPredictor", len(result) == 1)
        tmp.unlink(missing_ok=True)
    except Exception as e:
        check("PolymerPredictor", False, str(e))

    # 8. Visualization
    try:
        from reports.visualizations import ReportGenerator
        gen = ReportGenerator(str(ROOT / "reports" / "plots"))
        y = np.random.randn(50)
        p = y + np.random.randn(50) * 0.1
        gen.plot_pred_vs_actual(y, p, "test", save_name="_test_pred_vs_actual")
        plot = ROOT / "reports" / "plots" / "_test_pred_vs_actual.png"
        check("Visualization", plot.exists())
        plot.unlink(missing_ok=True)
    except Exception as e:
        check("Visualization", False, str(e))

    # Summary
    print(f"\n{'='*50}")
    print(f"  RESULTS: {passed}/{passed + failed} passed")
    print(f"{'='*50}")
    for name, status, detail in results:
        print(f"  [{status}] {name}")
        if detail:
            print(f"    {detail}")
    if failed == 0:
        print(f"\n  All tests passed!")
    else:
        print(f"\n  {failed} test(s) failed.")

test_notebook()

In [ ]:
#@title 22. Generate Final Reports
from pathlib import Path
ROOT = Path.cwd()
pred_dir = ROOT / "predictions"
if list(pred_dir.glob("*.pkl")):
    print("Generating reports...")
    !python reports/generate_reports.py --config config.yaml --skip-shap
else:
    print("No predictions. Run training first.")